In [1]:
#!pip install gymnasium numpy pandas yfinance scikit-learn

# Thư viện và cấu hình

In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CẤU HÌNH THÔNG SỐ THEO BÀI BÁO
# ==========================================
TICKERS = [
    '000002.SZ', '000166.SZ', '000568.SZ', 
    '000063.SZ', '000338.SZ', '000625.SZ', 
    '000100.SZ', '000538.SZ', '000725.SZ'
]  # Danh mục Biến động (Volatile Portfolio - Table A.5)

FEATURES = ['Open', 'High', 'Low', 'Close']  # F = 4 đặc trưng giá
WINDOW_SIZE = 30                              # M = 30 ngày (Sliding Window)
TRANSACTION_FEE = 0.0025                      # mu = 0.25% (Mục 4.1.1)

# Thời gian chia tập (Mục 4.1.2)
TRAIN_START, TRAIN_END = '2019-03-01', '2022-03-01'
TEST_START, TEST_END   = '2023-01-03', '2023-12-29'

# Tải dữ liệu và tiền xử lý

In [3]:
def fetch_and_preprocess_data(tickers, features, train_start, test_end):
    print(">>> Đang tải dữ liệu lịch sử từ Yahoo Finance...")
    # Tải dư về phía trước để có đủ window cho ngày đầu tiên
    fetch_start = pd.to_datetime(train_start) - pd.Timedelta(days=WINDOW_SIZE * 2)
    df_raw = yf.download(tickers, start=fetch_start, end=test_end, group_by='ticker', auto_adjust=False)
    
    # Chuẩn hóa format dataframe
    dfs = {}
    for ticker in tickers:
        df_t = df_raw[ticker][features].copy()
        # Xử lý giá trị thiếu: Forward-fill bằng giá ngày giao dịch trước đó (Mục 4.1.2)
        df_t = df_t.ffill().bfill()
        dfs[ticker] = df_t
    
    # Đồng bộ hóa ngày giao dịch chung cho tất cả các mã
    common_dates = dfs[tickers[0]].index
    for ticker in tickers[1:]:
        common_dates = common_dates.intersection(dfs[ticker].index)
    
    for ticker in tickers:
        dfs[ticker] = dfs[ticker].loc[common_dates]
        
    return dfs, common_dates

dfs, common_dates = fetch_and_preprocess_data(TICKERS, FEATURES, TRAIN_START, TEST_END)

# Tách tập Train và Test theo thời gian (Tránh Look-ahead bias)
train_dates = common_dates[(common_dates >= TRAIN_START) & (common_dates <= TRAIN_END)]
test_dates  = common_dates[(common_dates >= TEST_START) & (common_dates <= TEST_END)]

print(f"Số ngày giao dịch tập Train: {len(train_dates)} (Bài báo: ~727)")
print(f"Số ngày giao dịch tập Test:  {len(test_dates)} (Bài báo: ~242)")

>>> Đang tải dữ liệu lịch sử từ Yahoo Finance...


[*********************100%***********************]  9 of 9 completed

Số ngày giao dịch tập Train: 727 (Bài báo: ~727)
Số ngày giao dịch tập Test:  241 (Bài báo: ~242)


# CHUẨN HÓA & TẠO TENSOR QUAN SÁT o_t

In [4]:
def build_dataset_tensors(dfs, tickers, features, target_dates, all_dates, window_size=30, scaler=None):
    """
    Tạo mảng quan sát o_t có kích thước: (T, N, M, F)
    và vector giá đóng cửa thực tế để tính tỷ suất lợi nhuận y_t
    """
    if scaler is None:
        scaler = MinMaxScaler()
        # Fit scaler trên toàn bộ dữ liệu lịch sử của tập Train
        concat_train = np.concatenate([dfs[tic].loc[target_dates].values for tic in tickers], axis=0)
        scaler.fit(concat_train)
    
    N = len(tickers)
    M = window_size
    F = len(features)
    T = len(target_dates)
    
    observations = np.zeros((T, N, M, F), dtype=np.float32)
    closing_prices = np.zeros((T, N), dtype=np.float32)
    
    date_to_idx = {d: i for i, d in enumerate(all_dates)}
    
    for t_idx, d in enumerate(target_dates):
        curr_pos = date_to_idx[d]
        # Lấy window trượt [curr_pos - M + 1 : curr_pos + 1]
        for n_idx, tic in enumerate(tickers):
            window_slice = dfs[tic].iloc[curr_pos - M + 1 : curr_pos + 1].values
            # Min-Max Scaling (Mục 4.1.2)
            normalized_window = scaler.transform(window_slice)
            observations[t_idx, n_idx, :, :] = normalized_window
            closing_prices[t_idx, n_idx] = dfs[tic].iloc[curr_pos]['Close']
            
    return observations, closing_prices, scaler

# Chuẩn bị Tensor cho Train và Test
obs_train, prices_train, scaler = build_dataset_tensors(
    dfs, TICKERS, FEATURES, train_dates, common_dates, WINDOW_SIZE, scaler=None
)
obs_test, prices_test, _ = build_dataset_tensors(
    dfs, TICKERS, FEATURES, test_dates, common_dates, WINDOW_SIZE, scaler=scaler
)

print(f"Kích thước tensor quan sát Train o_t: {obs_train.shape} -> (T x N x M x F)")
print(f"Kích thước tensor quan sát Test  o_t: {obs_test.shape} -> (T x N x M x F)")

Kích thước tensor quan sát Train o_t: (727, 9, 30, 4) -> (T x N x M x F)
Kích thước tensor quan sát Test  o_t: (241, 9, 30, 4) -> (T x N x M x F)


# MÔI TRƯỜNG GIAO DỊCH FINRL

In [5]:

class PortfolioZeroEnv(gym.Env):
    """
    Môi trường mô phỏng giao dịch danh mục cổ phiếu dựa theo chuẩn FinRL / OpenAI Gym.
    Thực thi chặt chẽ các công thức (1) đến (6) trong bài báo:
    - N cổ phiếu + 1 tài sản phi rủi ro (Tiền mặt: index 0).
    - Vector trọng số w_t có tổng bằng 1.
    - Tính chi phí tái cân bằng với phí giao dịch mu = 0.0025.
    - Phần thưởng tức thời r_t = ln(p_t / p_{t-1}).
    """
    metadata = {'render.modes': ['human']}

    def __init__(self, observations, closing_prices, fee_rate=0.0025):
        super(PortfolioZeroEnv, self).__init__()
        self.obs = observations          # Tensor shape (T, N, M, F)
        self.prices = closing_prices      # Shape (T, N)
        self.fee_rate = fee_rate
        
        self.T, self.N, self.M, self.F = self.obs.shape
        self.num_assets = self.N + 1      # 1 Cash + 9 Stocks
        
        # Không gian hành động: Continuous action vector w_t in [0, 1]^(N+1), sum(w) = 1
        self.action_space = spaces.Box(
            low=0.0, high=1.0, shape=(self.num_assets,), dtype=np.float32
        )
        
        # Không gian quan sát: o_t in R^(N x M x F)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.N, self.M, self.F), dtype=np.float32
        )
        
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0
        self.portfolio_value = 1.0  # p_0 = 1.0
        
        # Trọng số khởi tạo: 100% tiền mặt ở thời điểm t=0
        self.w = np.zeros(self.num_assets, dtype=np.float32)
        self.w[0] = 1.0  # w_0^0 = 1.0, còn lại = 0
        
        return self.obs[self.t], {}

    def _calculate_cost_factor(self, w_new, w_old_prime):
        """
        Tính hệ số chiết khấu phí giao dịch mu_t (Ormos & Urban, Jiang et al.)
        Dựa trên độ chênh lệch trọng số giữa vị thế mong muốn w_t và vị thế trôi dạt w'_t.
        """
        # Tránh tính chi phí trên tiền mặt trực tiếp
        turnover = np.sum(np.abs(w_new[1:] - w_old_prime[1:]))
        mu_t = 1.0 - self.fee_rate * turnover
        return max(mu_t, 1e-6)

    def step(self, action):
        # 1. Đảm bảo ràng buộc simplex: w_i >= 0 và sum(w_i) = 1 (Công thức 2 & Mục 3.3.2)
        action = np.clip(action, 0.0, None)
        action_sum = np.sum(action)
        w_t = action / action_sum if action_sum > 0 else np.array([1.0] + [0.0]*self.N)
        
        self.t += 1
        
        # 2. Tính Vector biến động giá tương đối y_t = v_t / v_{t-1} (Công thức 1)
        # Cash luôn có y_t^0 = 1.0
        price_rel = self.prices[self.t] / (self.prices[self.t - 1] + 1e-8)
        y_t = np.concatenate([[1.0], price_rel])
        
        # 3. Trọng số tự trôi dạt w'_t theo biến động thị trường (Công thức 5)
        # w'_t = (y_t (.) w_{t-1}) / (y_t . w_{t-1})
        drift_w = (y_t * self.w) / (np.dot(y_t, self.w) + 1e-8)
        
        # 4. Tính toán phí giao dịch mu_t và cập nhật giá trị danh mục (Công thức 4)
        mu_t = self._calculate_cost_factor(w_t, drift_w)
        growth_factor = np.dot(y_t, self.w)
        
        p_prev = self.portfolio_value
        p_t = mu_t * p_prev * growth_factor
        self.portfolio_value = p_t
        
        # 5. Phần thưởng tức thời: Logarithmic Return r_t = ln(p_t / p_{t-1}) (Công thức 6)
        reward = float(np.log(max(p_t / p_prev, 1e-8)))
        
        # Cập nhật trọng số hiện tại cho bước kế tiếp
        self.w = w_t
        
        done = (self.t >= self.T - 1)
        terminated = done
        truncated = False
        
        next_obs = self.obs[self.t] if not done else self.obs[-1]
        
        info = {
            'portfolio_value': self.portfolio_value,
            'log_return': reward,
            'fee_factor': mu_t,
            'weights': self.w
        }
        
        return next_obs, reward, terminated, truncated, info


In [6]:
train_env = PortfolioZeroEnv(obs_train, prices_train, fee_rate=TRANSACTION_FEE)
test_env  = PortfolioZeroEnv(obs_test, prices_test, fee_rate=TRANSACTION_FEE)

# Test nhanh một vòng lặp reset & step
state, _ = train_env.reset()
print(f"\n>>> Kiểm tra môi trường:")
print(f"- Quan sát ban đầu s_0 kích thước: {state.shape}")

# Giả lập 1 action ngẫu nhiên từ Softmax (1 cash + 9 cổ phiếu)
dummy_action = np.ones(train_env.num_assets) / train_env.num_assets
next_state, reward, term, trunc, info = train_env.step(dummy_action)

print(f"- Thực hiện 1 bước rebalancing thành công:")
print(f"  + Trọng số phân bổ (10 assets): {np.round(info['weights'], 3)}")
print(f"  + Log Return (Reward r_t): {reward:.6f}")
print(f"  + Giá trị danh mục p_1: {info['portfolio_value']:.6f}")
print(f"  + Hệ số chi phí mu_1: {info['fee_factor']:.6f}")
print(">>> Bước 1 đã hoàn tất thành công và sẵn sàng tích hợp với Representation Network (Transformer) ở Bước 2.")


>>> Kiểm tra môi trường:
- Quan sát ban đầu s_0 kích thước: (9, 30, 4)
- Thực hiện 1 bước rebalancing thành công:
  + Trọng số phân bổ (10 assets): [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
  + Log Return (Reward r_t): -0.002253
  + Giá trị danh mục p_1: 0.997750
  + Hệ số chi phí mu_1: 0.997750
>>> Bước 1 đã hoàn tất thành công và sẵn sàng tích hợp với Representation Network (Transformer) ở Bước 2.


# Thiết lập Representation Network
## 1. CHUẨN BỊ DATASET CHO TIỀN HUẤN LUYỆN (D=1 và D=5)

In [7]:
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Thiết lập thiết bị tính toán (GPU nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> Thiết bị đang sử dụng: {device}")


class PretrainDataset(Dataset):
    """
    Tạo các cặp (o_enc, o_dec_1, y_1, o_dec_5, y_5) phục vụ 2 tác vụ pre-train D=1 và D=5.
    Mục tiêu y là giá Close đã được chuẩn hóa của các ngày tiếp theo.
    """
    def __init__(self, obs_data, max_d=5, close_idx=3):
        self.obs = obs_data  # Shape: (T, N, M, F)
        self.max_d = max_d
        self.close_idx = close_idx
        self.valid_length = len(obs_data) - max_d

    def __len__(self):
        return self.valid_length

    def __getitem__(self, idx):
        # 1. Đầu vào Encoder: o_enc có shape (N, M, F)
        o_enc = self.obs[idx]
        
        # 2. Tác vụ D = 1
        # o_dec_1 lấy đặc trưng của ngày tiếp theo: shape (N, 1, F)
        o_dec_1 = self.obs[idx + 1, :, -1:, :]  
        # y_1 là giá Close ngày tiếp theo: shape (N, 1)
        y_1 = self.obs[idx + 1, :, -1, self.close_idx]  
        
        # 3. Tác vụ D = 5
        # o_dec_5 lấy đặc trưng 5 ngày tiếp theo: shape (N, 5, F)
        # Ghép bước giá cuối cùng của 5 ngày liên tiếp [idx+1 ... idx+5]
        future_steps = [self.obs[idx + d, :, -1:, :] for d in range(1, 6)]
        o_dec_5 = np.concatenate(future_steps, axis=1)
        # y_5 là giá Close của 5 ngày tiếp theo: shape (N, 5)
        y_5 = np.stack([self.obs[idx + d, :, -1, self.close_idx] for d in range(1, 6)], axis=-1)

        return (
            torch.tensor(o_enc, dtype=torch.float32),
            torch.tensor(o_dec_1, dtype=torch.float32),
            torch.tensor(y_1, dtype=torch.float32),
            torch.tensor(o_dec_5, dtype=torch.float32),
            torch.tensor(y_5, dtype=torch.float32)
        )

# Tạo DataLoader cho tập Train (In-sample: 2019-03-01 đến 2022-03-01)
pretrain_dataset = PretrainDataset(obs_train, max_d=5, close_idx=FEATURES.index('Close'))
pretrain_loader = DataLoader(pretrain_dataset, batch_size=32, shuffle=True)



>>> Thiết bị đang sử dụng: cuda


## 2. KIẾN TRÚC TRANSFORMER ENCODER - DECODER

In [8]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (Batch * N, Seq_Len, d_model)
        return x + self.pe[:, :x.size(1), :]

class PortfolioTransformer(nn.Module):
    """
    Kiến trúc Transformer Encoder-Decoder (Mục 3.3.1)
    - Encoder: Nhúng dữ liệu giá theo chiều thời gian để trích xuất temporal dynamics.
    - Decoder: Dự đoán giá tương lai qua Cross-Attention.
    """
    def __init__(self, feature_dim=4, hidden_dim=64, nhead=4, num_layers=2):
        super(PortfolioTransformer, self).__init__()
        self.hidden_dim = hidden_dim
        self.feature_dim = feature_dim
        
        # 1. Các khối của Encoder
        self.enc_input_fc = nn.Linear(feature_dim, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim, max_len=100)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim * 2, 
            batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 2. Các khối của Decoder
        self.dec_input_fc = nn.Linear(feature_dim, hidden_dim)
        self.pos_decoder = PositionalEncoding(hidden_dim, max_len=100)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim * 2, 
            batch_first=True, dropout=0.1
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        # 3. Đầu ra dự báo giá cổ phiếu
        self.pred_head = nn.Linear(hidden_dim, 1)

    def encode(self, o_enc):
        """
        Khối Encoder h_theta(o_t)
        o_enc: (B, N, M, F) -> s_enc: (B, N, M, H)
        """
        B, N, M, F = o_enc.shape
        x = o_enc.view(B * N, M, F)
        x = self.enc_input_fc(x)
        x = self.pos_encoder(x)
        s_enc = self.encoder(x)
        return s_enc.view(B, N, M, self.hidden_dim)

    def decode(self, o_dec, s_enc):
        """
        o_dec: (B, N, D, F), s_enc: (B, N, M, H) -> y_hat: (B, N, D)
        """
        B, N, D, F = o_dec.shape
        M = s_enc.shape[2]
        
        tgt = o_dec.view(B * N, D, F)
        memory = s_enc.view(B * N, M, self.hidden_dim)
        
        tgt = self.dec_input_fc(tgt)
        tgt = self.pos_decoder(tgt)
        
        # Cross Attention giữa target tương lai và hidden state từ Encoder
        dec_out = self.decoder(tgt=tgt, memory=memory)
        y_hat = self.pred_head(dec_out).squeeze(-1)  # (B*N, D)
        return y_hat.view(B, N, D)

    def forward(self, o_enc, o_dec):
        s_enc = self.encode(o_enc)
        y_hat = self.decode(o_dec, s_enc)
        return y_hat, s_enc



## 3. TIỀN HUẤN LUYỆN (PRE-TRAINING PHASE - Công thức 13)

In [9]:
def train_representation_model(model, dataloader, epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    print(">>> Bắt đầu giai đoạn Tiền huấn luyện (Pre-training Phase)...")
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        loss_d1_accum = 0.0
        loss_d5_accum = 0.0
        
        for o_enc, o_dec_1, y_1, o_dec_5, y_5 in dataloader:
            o_enc = o_enc.to(device)
            o_dec_1, y_1 = o_dec_1.to(device), y_1.to(device)
            o_dec_5, y_5 = o_dec_5.to(device), y_5.to(device)
            
            optimizer.zero_grad()
            
            # --- Task 1: Dự đoán giá 1 ngày tiếp theo (D = 1) ---
            s_enc = model.encode(o_enc)
            y_hat_1 = model.decode(o_dec_1, s_enc).squeeze(-1) # (B, N)
            # Hàm mất mát MSE L_P_1 (Công thức 13: 1 / (D * N) * sum(...))
            loss_d1 = torch.mean((y_1 - y_hat_1) ** 2)
            
            # --- Task 2: Dự đoán giá 5 ngày tiếp theo (D = 5) ---
            y_hat_5 = model.decode(o_dec_5, s_enc)             # (B, N, 5)
            # Hàm mất mát MSE L_P_5 (Công thức 13: 1 / (D * N) * sum(...))
            loss_d5 = torch.mean((y_5 - y_hat_5) ** 2)
            
            # Tổng hợp hàm mất mát của 2 tác vụ pre-train
            loss = loss_d1 + loss_d5
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            loss_d1_accum += loss_d1.item()
            loss_d5_accum += loss_d5.item()
            
        if epoch % 5 == 0 or epoch == 1:
            avg_loss = total_loss / len(dataloader)
            print(f"Epoch [{epoch:02d}/{epochs:02d}] - Total MSE Loss: {avg_loss:.6f} "
                  f"(L_P_1: {loss_d1_accum/len(dataloader):.6f}, L_P_5: {loss_d5_accum/len(dataloader):.6f})")

# Khởi tạo mô hình và huấn luyện
H_DIM = 64  # Hidden dimension H
model = PortfolioTransformer(feature_dim=len(FEATURES), hidden_dim=H_DIM, nhead=4, num_layers=2).to(device)
train_representation_model(model, pretrain_loader, epochs=20, lr=1e-3)




>>> Bắt đầu giai đoạn Tiền huấn luyện (Pre-training Phase)...
Epoch [01/20] - Total MSE Loss: 0.388160 (L_P_1: 0.188522, L_P_5: 0.199638)
Epoch [05/20] - Total MSE Loss: 0.009773 (L_P_1: 0.004974, L_P_5: 0.004799)
Epoch [10/20] - Total MSE Loss: 0.003743 (L_P_1: 0.001881, L_P_5: 0.001862)
Epoch [15/20] - Total MSE Loss: 0.002303 (L_P_1: 0.001190, L_P_5: 0.001113)
Epoch [20/20] - Total MSE Loss: 0.001651 (L_P_1: 0.000855, L_P_5: 0.000796)


## 4. TRÍCH XUẤT ENCODER h_theta(o_t) ĐỂ PHỤC VỤ RL

In [10]:


class RepresentationNetwork(nn.Module):
    """
    Module đại diện h_theta(o_t) trích xuất từ Transformer sau pre-training.
    Nhận o_t in R^(N x M x F) -> Trả về trạng thái ẩn s_t in R^(N x M x H)
    Được sử dụng trực tiếp trong Bước 3 & Algorithm 1.
    """
    def __init__(self, pretrained_model):
        super(RepresentationNetwork, self).__init__()
        # Kế thừa trọng số Encoder đã qua tiền huấn luyện
        self.enc_input_fc = pretrained_model.enc_input_fc
        self.pos_encoder = pretrained_model.pos_encoder
        self.encoder = pretrained_model.encoder
        self.hidden_dim = pretrained_model.hidden_dim

    def forward(self, o_t):
        """
        o_t: tensor shape (N, M, F) hoặc batch (B, N, M, F)
        """
        has_batch = (o_t.dim() == 4)
        if not has_batch:
            o_t = o_t.unsqueeze(0)  # Thêm Batch dim -> (1, N, M, F)
            
        B, N, M, F = o_t.shape
        x = o_t.view(B * N, M, F)
        x = self.enc_input_fc(x)
        x = self.pos_encoder(x)
        s_t = self.encoder(x)
        s_t = s_t.view(B, N, M, self.hidden_dim)
        
        return s_t if has_batch else s_t.squeeze(0)

# Trích xuất Encoder
h_theta = RepresentationNetwork(model).to(device)

# --- Kiểm tra chuyển đổi trạng thái từ Bước 1 sang Bước 2 ---
sample_obs, _ = train_env.reset()  # Lấy quan sát o_0 từ môi trường ở Bước 1: shape (9, 30, 4)
sample_obs_tensor = torch.tensor(sample_obs, dtype=torch.float32).to(device)

with torch.no_grad():
    s_0 = h_theta(sample_obs_tensor)

print("\n>>> Hoàn tất Bước 2:")
print(f"- Quan sát thực tế đầu vào o_0:   {sample_obs_tensor.shape} (N x M x F)")
print(f"- Trạng thái ẩn đầu ra s_0:       {s_0.shape} (N x M x H) với H={H_DIM}")
print(">>> Mô-đun h_theta đã sẵn sàng chuyển giao cho Prediction Network & Dynamics Network ở Bước 3.")


>>> Hoàn tất Bước 2:
- Quan sát thực tế đầu vào o_0:   torch.Size([9, 30, 4]) (N x M x F)
- Trạng thái ẩn đầu ra s_0:       torch.Size([9, 30, 64]) (N x M x H) với H=64
>>> Mô-đun h_theta đã sẵn sàng chuyển giao cho Prediction Network & Dynamics Network ở Bước 3.


# Dựng Prediction Network & Dynamics Network

## 1. PREDICTION NETWORK f_theta(s_t) (Mục 3.3.2)

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PredictionNetwork(nn.Module):
    """
    Prediction Network f_theta:
    - Đầu vào: s_t in R^(N x M x H)
    - Đầu ra:
        + a_t (Action/Weights): Vector tỷ trọng 10 tài sản qua Softmax.
        + v_t (Value): Điểm đánh giá trạng thái hiện tại qua 2 lớp FC.
    """
    def __init__(self, num_assets=10, N=9, M=30, H=64, hidden_dim=256):
        super(PredictionNetwork, self).__init__()
        self.N = N
        self.M = M
        self.H = H
        self.state_flat_dim = N * M * H
        
        # Khối trích xuất đặc trưng chung từ không gian ẩn
        self.feature_extractor = nn.Sequential(
            nn.Linear(self.state_flat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # Policy Head -> Output Action a_t (1 cash + 9 stocks)
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_assets)
        )
        
        # Value Head -> 2 lớp Fully Connected để ra điểm vô hướng (scalar)
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, s_t):
        """
        s_t: shape (N, M, H) hoặc batch (B, N, M, H)
        """
        has_batch = (s_t.dim() == 4)
        if not has_batch:
            s_t = s_t.unsqueeze(0)  # Thêm Batch dim -> (1, N, M, H)
            
        B = s_t.shape[0]
        s_flat = s_t.reshape(B, -1)
        features = self.feature_extractor(s_flat)
        
        # Action a_t: Đi qua Softmax chuẩn hóa tỷ trọng (sum = 1, w_i in [0, 1])
        action_logits = self.policy_head(features)
        a_t = F.softmax(action_logits, dim=-1)
        
        # Value v_t: Đánh giá giá trị trạng thái (scalar)
        v_t = self.value_head(features)
        
        if not has_batch:
            return a_t.squeeze(0), v_t.squeeze(0)
        return a_t, v_t




# 2. DYNAMICS NETWORK g_theta(s_t, a_t) (Mục 3.3.3)

In [12]:

class DynamicsNetwork(nn.Module):
    """
    Dynamics Network g_theta:
    - Đầu vào: s_t in R^(N x M x H) và a_t in R^(num_assets)
    - Đầu ra:
        + s_{t+1}: Trạng thái ẩn kế tiếp cùng kích thước (N x M x H)
        + r_t: Phần thưởng tức thời dự đoán (scalar)
    """
    def __init__(self, num_assets=10, N=9, M=30, H=64, hidden_dim=256):
        super(DynamicsNetwork, self).__init__()
        self.N = N
        self.M = M
        self.H = H
        self.state_flat_dim = N * M * H
        
        # Khối kết hợp trạng thái ẩn s_t và hành động a_t
        self.state_action_fc = nn.Sequential(
            nn.Linear(self.state_flat_dim + num_assets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # Khối chuyển tiếp trạng thái tiếp theo (Next State Head)
        self.next_state_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.state_flat_dim)
        )
        
        # Khối dự đoán phần thưởng tức thì (Immediate Reward Head)
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, s_t, a_t):
        """
        s_t: shape (N, M, H) hoặc batch (B, N, M, H)
        a_t: shape (num_assets,) hoặc batch (B, num_assets)
        """
        has_batch = (s_t.dim() == 4)
        if not has_batch:
            s_t = s_t.unsqueeze(0)
            a_t = a_t.unsqueeze(0)
            
        B = s_t.shape[0]
        s_flat = s_t.reshape(B, -1)
        
        # Kết hợp trạng thái và hành động
        sa_cat = torch.cat([s_flat, a_t], dim=-1)
        latent_dyn = self.state_action_fc(sa_cat)
        
        # 1. Dự đoán trạng thái ẩn tiếp theo s_{t+1}
        # Có thể thêm residual connection để huấn luyện ổn định hơn: s_{t+1} = s_t + delta_s
        delta_s = self.next_state_head(latent_dyn)
        next_s_flat = s_flat + delta_s
        s_next = next_s_flat.reshape(B, self.N, self.M, self.H)
        
        # 2. Dự đoán phần thưởng tức thì r_t (scalar)
        r_t = self.reward_head(latent_dyn)
        
        if not has_batch:
            return r_t.squeeze(0), s_next.squeeze(0)
        return r_t, s_next




## 3. KHỞI TẠO VÀ KIỂM TRA MÔ HÌNH VỚI DỮ LIỆU BƯỚC TRƯỚC

In [13]:

N_ASSETS = train_env.num_assets  # 10 tài sản
N_STOCKS = len(TICKERS)         # 9 cổ phiếu
M_WINDOW = WINDOW_SIZE          # 30 ngày
H_FEATURE = H_DIM               # 64 chiều ẩn từ Bước 2

# Khởi tạo hai mô hình
f_theta = PredictionNetwork(num_assets=N_ASSETS, N=N_STOCKS, M=M_WINDOW, H=H_FEATURE).to(device)
g_theta = DynamicsNetwork(num_assets=N_ASSETS, N=N_STOCKS, M=M_WINDOW, H=H_FEATURE).to(device)

print(">>> Đã khởi tạo thành công Prediction Network và Dynamics Network.")

# --- KIỂM TRA DÒNG DỮ LIỆU ĐỒNG BỘ TỪ s_0 Ở BƯỚC 2 ---
# 1. Kiểm tra Prediction Network: s_0 -> (a_0, v_0)
a_0, v_0 = f_theta(s_0)

print("\n>>> Kiểm tra luồng tính toán Prediction Network:")
print(f"- Đầu vào s_0: shape {s_0.shape}")
print(f"- Hành động đầu ra a_0: shape {a_0.shape}")
print(f"  + Giá trị từng trọng số: {np.round(a_0.detach().cpu().numpy(), 4)}")
print(f"  + Tổng trọng số (sum of weights): {torch.sum(a_0).item():.6f} (Phải = 1.0)")
print(f"  + Kiểm tra chặn [0, 1]: {(a_0 >= 0).all().item() and (a_0 <= 1).all().item()}")
print(f"- Giá trị trạng thái v_0: {v_0.item():.6f} (Scalar)")

# 2. Kiểm tra Dynamics Network: (s_0, a_0) -> (r_0, s_1)
r_0, s_1 = g_theta(s_0, a_0)

print("\n>>> Kiểm tra luồng tính toán Dynamics Network (Internal Simulator):")
print(f"- Đầu vào: s_0 {s_0.shape} và a_0 {a_0.shape}")
print(f"- Phần thưởng nội bộ dự đoán r_0: {r_0.item():.6f} (Scalar)")
print(f"- Trạng thái ẩn kế tiếp dự đoán s_1: shape {s_1.shape} (Phải khớp chính xác với s_0)")
print(">>> Bước 3 hoàn thành. Bộ đôi f_theta và g_theta đã sẵn sàng kết hợp với MCTS continuous & Noise OU ở Bước 4.")

>>> Đã khởi tạo thành công Prediction Network và Dynamics Network.

>>> Kiểm tra luồng tính toán Prediction Network:
- Đầu vào s_0: shape torch.Size([9, 30, 64])
- Hành động đầu ra a_0: shape torch.Size([10])
  + Giá trị từng trọng số: [0.0813 0.1025 0.0912 0.0994 0.1199 0.1097 0.1007 0.1109 0.0933 0.091 ]
  + Tổng trọng số (sum of weights): 1.000000 (Phải = 1.0)
  + Kiểm tra chặn [0, 1]: True
- Giá trị trạng thái v_0: -0.170286 (Scalar)

>>> Kiểm tra luồng tính toán Dynamics Network (Internal Simulator):
- Đầu vào: s_0 torch.Size([9, 30, 64]) và a_0 torch.Size([10])
- Phần thưởng nội bộ dự đoán r_0: 0.106499 (Scalar)
- Trạng thái ẩn kế tiếp dự đoán s_1: shape torch.Size([9, 30, 64]) (Phải khớp chính xác với s_0)
>>> Bước 3 hoàn thành. Bộ đôi f_theta và g_theta đã sẵn sàng kết hợp với MCTS continuous & Noise OU ở Bước 4.


# Tích hợp MCTS cho Không gian Hành động Liên tục
## 1. BỘ SINH NHIỄU ORNSTEIN-UHLENBECK (OU NOISE) (Mục 3.3.4)

In [14]:
import torch
import torch.nn.functional as F
import numpy as np

class OUNoise:
    """
    Tiến trình ngẫu nhiên Ornstein-Uhlenbeck tạo nhiễu tương quan thời gian,
    giúp mở rộng không gian khám phá hành động liên tục [Uhlenbeck & Ornstein, 1930].
    dx_t = theta * (mu - x_t) * dt + sigma * dW_t
    """
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.size = size
        self.mu = mu * np.ones(self.size)
        self.theta = theta
        self.sigma = sigma
        self.reset()

    def reset(self):
        self.state = np.copy(self.mu)

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.size)
        self.state += dx
        return torch.tensor(self.state, dtype=torch.float32)



##  2. CẤU TRÚC NODE & CÂY TÌM KIẾM MCTS TRONG KHÔNG GIAN ẨN

In [15]:

class MCTSNode:
    """
    Node trong cây MCTS đại diện cho trạng thái ẩn s.
    Mỗi node lưu các nhánh hành động dẫn tới trạng thái kế tiếp.
    """
    def __init__(self, state, prior_prob=1.0):
        self.state = state                    # Hidden state tensor s in R^(N x M x H)
        self.prior_prob = prior_prob          # Xác suất tiên nghiệm P(s, a)
        self.is_expanded = False
        
        self.children = {}                    # Dict: action_idx -> MCTSNode
        self.actions = []                     # Danh sách các action tensor a_i
        self.rewards = {}                     # Dict: action_idx -> predicted reward r
        
        # Thống kê tìm kiếm
        self.visit_count = 0                  # N(s)
        self.action_visits = {}               # N(s, a)
        self.action_q_values = {}             # Q(s, a)

    def expand(self, actions, dynamics_net):
        """
        Giai đoạn Expansion:
        Dùng Dynamics Network g_theta để sinh iota trạng thái con và phần thưởng tức thì.
        """
        self.actions = actions
        with torch.no_grad():
            for idx, a in enumerate(actions):
                # Dự đoán trạng thái con kế tiếp và reward nội bộ qua Dynamics Network
                pred_r, s_next = dynamics_net(self.state, a)
                self.rewards[idx] = pred_r.item()
                self.children[idx] = MCTSNode(state=s_next, prior_prob=1.0 / len(actions))
                self.action_visits[idx] = 0
                self.action_q_values[idx] = 0.0
                
        self.is_expanded = True

    def select_action_idx(self, c_puct=1.25):
        """
        Giai đoạn Selection: Áp dụng công thức UCB/PUCT để chọn nhánh tối ưu.
        a* = argmax [ Q(s, a) + c_puct * P(s, a) * sqrt(sum N) / (1 + N(s, a)) ]
        """
        best_score = -float('inf')
        best_idx = 0
        total_visits = sum(self.action_visits.values())

        for idx in range(len(self.actions)):
            n_sa = self.action_visits[idx]
            q_sa = self.action_q_values[idx]
            p_sa = 1.0 / len(self.actions)
            
            u_score = c_puct * p_sa * (np.sqrt(total_visits) / (1.0 + n_sa))
            score = q_sa + u_score
            
            if score > best_score:
                best_score = score
                best_idx = idx
                
        return best_idx



##  3. THUẬT TOÁN MCTS CHO PORTFOLIOZERO (Mục 3.3.4 & Algorithm 1)

In [16]:

class ContinuousMCTS:
    """
    Bộ tìm kiếm Monte Carlo Tree Search chuyên biệt cho danh mục PortfolioZero:
    - Tiếp nhận s_t.
    - Tạo iota = 4 hành động (1 deterministic từ f_theta + 3 perturbed bằng OU noise).
    - Trả về hành động có số lượt ghé thăm cao nhất: hat{a}_t
    """
    def __init__(self, pred_net, dyn_net, iota=4, num_simulations=25, gamma=0.99, c_puct=1.25):
        self.pred_net = pred_net
        self.dyn_net = dyn_net
        self.iota = iota                      # iota = 4 theo bài báo
        self.num_simulations = num_simulations
        self.gamma = gamma
        self.c_puct = c_puct
        self.ou_process = OUNoise(size=pred_net.policy_head[-1].out_features)

    def _generate_candidate_actions(self, s_t):
        """
        Tạo iota = 4 hành động:
        + Hành động 0: a_deterministic từ f_theta(s_t)
        + Hành động 1, 2, 3: cộng thêm OU noise độc lập, chuẩn hóa qua Simplex
        """
        with torch.no_grad():
            a_base, _ = self.pred_net(s_t)
            
        candidate_actions = [a_base]
        
        # Tạo iota - 1 = 3 hành động thêm bằng cách cộng OU Noise
        for _ in range(self.iota - 1):
            noise = self.ou_process.sample().to(s_t.device)
            # Thêm nhiễu và chiếu về không gian tỷ trọng hợp lệ (w_i >= 0, sum w_i = 1)
            perturbed_a = torch.clamp(a_base + noise, min=1e-5)
            perturbed_a = perturbed_a / torch.sum(perturbed_a)
            candidate_actions.append(perturbed_a)
            
        return candidate_actions

    def search(self, root_state):
        """
        Quy trình mô phỏng MCTS (Selection, Expansion, Backpropagation)
        """
        root_node = MCTSNode(state=root_state)
        
        # 1. Tạo tập iota = 4 hành động tại gốc
        root_actions = self._generate_candidate_actions(root_state)
        root_node.expand(root_actions, self.dyn_net)
        
        # 2. Vòng lặp mô phỏng
        for _ in range(self.num_simulations):
            node = root_node
            search_path = [node]
            action_path = []
            
            # --- PHASE 1: SELECTION ---
            while node.is_expanded and len(node.children) > 0:
                action_idx = node.select_action_idx(self.c_puct)
                action_path.append(action_idx)
                node = node.children[action_idx]
                search_path.append(node)
                
            # --- PHASE 2: EXPANSION & EVALUATION (Tại lá) ---
            with torch.no_grad():
                leaf_a, leaf_v = self.pred_net(node.state)
                leaf_value = leaf_v.item()
                
            # Nếu node lá chưa mở rộng và độ sâu phù hợp, tiến hành mở rộng
            if not node.is_expanded:
                leaf_actions = self._generate_candidate_actions(node.state)
                node.expand(leaf_actions, self.dyn_net)
                
            # --- PHASE 3: BACKPROPAGATION ---
            # Truy hồi ngược từ lá về gốc để cập nhật N và Q
            g_return = leaf_value
            for i in reversed(range(len(action_path))):
                parent_node = search_path[i]
                act_idx = action_path[i]
                reward = parent_node.rewards[act_idx]
                
                # Chiết khấu giá trị theo công thức Bellman nội bộ
                g_return = reward + self.gamma * g_return
                
                parent_node.visit_count += 1
                parent_node.action_visits[act_idx] += 1
                n_count = parent_node.action_visits[act_idx]
                q_old = parent_node.action_q_values[act_idx]
                
                # Cập nhật trung bình động cho giá trị Q(s, a)
                parent_node.action_q_values[act_idx] = q_old + (g_return - q_old) / n_count

        # 3. Chọn hành động được ghé thăm nhiều nhất làm target action hat{a}_t
        visits = [root_node.action_visits[i] for i in range(self.iota)]
        best_action_idx = int(np.argmax(visits))
        target_a_t = root_node.actions[best_action_idx]
        
        return target_a_t, visits, root_node.actions



## 4. KIỂM THỬ TÍCH HỢP MCTS VỚI TRẠNG THÁI ẨN s_0

In [17]:

mcts_agent = ContinuousMCTS(
    pred_net=f_theta, 
    dyn_net=g_theta, 
    iota=4,                 # iota = 4 theo bài báo
    num_simulations=20,     # Số lượt mô phỏng cây
    gamma=0.99
)

print(">>> Bắt đầu chạy MCTS trên trạng thái ẩn s_0...")
target_a_0, visit_counts, candidates = mcts_agent.search(s_0)

print("\n>>> Kết quả phân tích MCTS cho không gian liên tục:")
print(f"- Số lượng nhánh hành động khởi tạo: iota = {len(candidates)}")
for idx, (a_cand, v_cnt) in enumerate(zip(candidates, visit_counts)):
    label = "Gốc (Deterministic f_theta)" if idx == 0 else f"Nhiễu OU số {idx}"
    print(f"  + Nhánh {idx} [{label}]: Số lượt ghé thăm (Visits) = {v_cnt} | Max weight: {torch.max(a_cand):.3f}")

print(f"\n- Hành động mục tiêu tối ưu được chọn (hat{{a}}_0) có shape: {target_a_0.shape}")
print(f"- Phân bổ trọng số của hat{{a}}_0: \n{np.round(target_a_0.detach().cpu().numpy(), 4)}")
print(f"- Tổng trọng số kiểm tra: {torch.sum(target_a_0).item():.6f}")
print(">>> Bước 4 hoàn thành thành công. hat{a}_t đã sẵn sàng đóng vai trò mục tiêu huấn luyện (target) cho Bước 5.")

>>> Bắt đầu chạy MCTS trên trạng thái ẩn s_0...

>>> Kết quả phân tích MCTS cho không gian liên tục:
- Số lượng nhánh hành động khởi tạo: iota = 4
  + Nhánh 0 [Gốc (Deterministic f_theta)]: Số lượt ghé thăm (Visits) = 5 | Max weight: 0.120
  + Nhánh 1 [Nhiễu OU số 1]: Số lượt ghé thăm (Visits) = 5 | Max weight: 0.173
  + Nhánh 2 [Nhiễu OU số 2]: Số lượt ghé thăm (Visits) = 5 | Max weight: 0.280
  + Nhánh 3 [Nhiễu OU số 3]: Số lượt ghé thăm (Visits) = 5 | Max weight: 0.311

- Hành động mục tiêu tối ưu được chọn (hat{a}_0) có shape: torch.Size([10])
- Phân bổ trọng số của hat{a}_0: 
[0.0813 0.1025 0.0912 0.0994 0.1199 0.1097 0.1007 0.1109 0.0933 0.091 ]
- Tổng trọng số kiểm tra: 1.000000
>>> Bước 4 hoàn thành thành công. hat{a}_t đã sẵn sàng đóng vai trò mục tiêu huấn luyện (target) cho Bước 5.


# Vòng lặp Huấn luyện &amp; Hàm Mất mát (Training Loop - Algorithm 1)

## 1. BỘ NHỚ REPLAY BUFFER B (Algorithm 1, Dòng 2 & 11)

In [18]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReplayBuffer:
    """
    Lưu trữ các Episode tương tác: (o_1, a_1, hat{r}_1, ..., o_T, a_T, hat{r}_T).
    Tính sẵn giá trị chiết khấu thực tế hat{v}_t để phục vụ huấn luyện Value.
    """
    def __init__(self, capacity=100):
        self.capacity = capacity
        self.buffer = []

    def push(self, trajectory):
        """
        trajectory: dict chứa 'obs', 'actions', 'rewards', 'returns'
        """
        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)
        self.buffer.append(trajectory)

    def sample_trajectory(self):
        return random.choice(self.buffer)

    def __len__(self):
        return len(self.buffer)


## 2. HÀM MẤT MÁT TỔNG HỢP JOINT LOSS (Công thức 14)

In [19]:

def compute_joint_loss(pred_a, target_a, pred_v, target_v, pred_r, target_r):
    """
    L_t(theta) = CE(a_k, hat{a}_k) + MSE(v_k, hat{v}_k) + MSE(r_k, hat{r}_k)
    """
    # 1. Cross-Entropy Loss cho phân bổ xác suất tỷ trọng danh mục
    # pred_a và target_a đều nằm trên Simplex (sum = 1, >= 0)
    loss_a = -torch.sum(target_a * torch.log(pred_a + 1e-8))
    
    # 2. MSE Loss cho dự đoán State Value
    loss_v = F.mse_loss(pred_v.view(-1), target_v.view(-1))
    
    # 3. MSE Loss cho dự đoán Immediate Reward của Dynamics Network
    loss_r = F.mse_loss(pred_r.view(-1), target_r.view(-1))
    
    total_step_loss = loss_a + loss_v + loss_r
    return total_step_loss, loss_a.item(), loss_v.item(), loss_r.item()



## 3. THUẬT TOÁN HUẤN LUYỆN PORTFOLIOZERO (Algorithm 1)

In [20]:

import time
import os

# ==========================================================
# CẤU HÌNH THAM SỐ CHUẨN BÀI BÁO (FULL TRAINING SETTINGS)
# ==========================================================
FULL_EPOCHS = 50           # Số epoch huấn luyện (50 - 100)
UNROLL_K = 5               # Số bước unroll K = 5 (Chuẩn MuZero / Công thức 14)
MCTS_SIMS = 30             # Số lần mô phỏng cây mỗi bước
GAMMA = 0.99               # Hệ số chiết khấu
LEARNING_RATE = 1e-4       # Learning rate eta
L2_REG = 1e-4              # Hệ số C cho L2 Regularization (C ||theta||^2)
SAMPLE_STEPS_PER_EPOCH = 64 # Số điểm thời gian lấy mẫu cập nhật loss mỗi epoch

# Cập nhật số simulation cho agent MCTS
mcts_agent.num_simulations = MCTS_SIMS

def train_full_portfoliozero(
    env, h_net, f_net, g_net, mcts,
    epochs=FULL_EPOCHS, K=UNROLL_K, gamma=GAMMA,
    lr=LEARNING_RATE, l2_reg=L2_REG, sample_steps=SAMPLE_STEPS_PER_EPOCH,
    save_path="best_portfoliozero_r1.pth"
):
    all_params = list(h_net.parameters()) + list(f_net.parameters()) + list(g_net.parameters())
    optimizer = torch.optim.Adam(all_params, lr=lr, weight_decay=l2_reg)
    replay_buffer = ReplayBuffer(capacity=20)
    
    best_return = -float('inf')
    total_T = env.T - 1  # Toàn bộ ~727 ngày giao dịch của tập Train
    
    print("=" * 75)
    print(f"BẮT ĐẦU HUẤN LUYỆN TOÀN DIỆN PORTFOLIOZERO R1 (FULL IN-SAMPLE PERIOD)")
    print(f"- Tổng số ngày giao dịch (T): {total_T} ngày")
    print(f"- Số Epochs: {epochs} | Unroll K: {K} | MCTS Sims: {mcts.num_simulations}")
    print(f"- Tối ưu hóa: Adam (lr={lr}, weight_decay={l2_reg})")
    print("=" * 75)

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        
        # ----------------------------------------------------
        # 1. EXECUTION PHASE: Chạy trọn vẹn chu kỳ T (Algorithm 1: lines 4-11)
        # ----------------------------------------------------
        obs_raw, _ = env.reset()
        h_net.eval(); f_net.eval(); g_net.eval()
        
        trajectory = {'obs': [], 'actions': [], 'rewards': []}
        obs_tensor = torch.tensor(obs_raw, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            s_t = h_net(obs_tensor)
            
        done = False
        step_count = 0
        
        while not done:
            # Dòng 7: Lấy hành động mục tiêu từ MCTS
            target_a_t, _, _ = mcts.search(s_t)
            action_np = target_a_t.detach().cpu().numpy()
            
            # Dòng 9: Thực thi trên thị trường, thu hoạch hat{r}_t = ln(p_t / p_{t-1})
            next_obs_raw, reward, terminated, truncated, _ = env.step(action_np)
            
            trajectory['obs'].append(obs_tensor.cpu())
            trajectory['actions'].append(target_a_t.cpu())
            trajectory['rewards'].append(reward)
            
            # Dòng 8: Dynamics network cập nhật trạng thái ẩn kế tiếp
            with torch.no_grad():
                _, s_t = g_net(s_t, target_a_t)
                
            obs_tensor = torch.tensor(next_obs_raw, dtype=torch.float32).to(device)
            step_count += 1
            done = terminated or truncated

        # Dòng 19: Tính tổng phần thưởng chiết khấu thực tế hat{v}_t
        T = len(trajectory['rewards'])
        returns = np.zeros(T, dtype=np.float32)
        running_ret = 0.0
        for i in reversed(range(T)):
            running_ret = trajectory['rewards'][i] + gamma * running_ret
            returns[i] = running_ret
        trajectory['returns'] = [torch.tensor(r, dtype=torch.float32) for r in returns]
        
        replay_buffer.push(trajectory)
        total_epoch_return = np.sum(trajectory['rewards'])
        
        # ----------------------------------------------------
        # 2. TRAINING PHASE: Cập nhật Joint Loss (Algorithm 1: lines 12-26)
        # ----------------------------------------------------
        h_net.train(); f_net.train(); g_net.train()
        trace = replay_buffer.sample_trajectory()
        T_trace = len(trace['rewards'])
        
        # Lấy mẫu các điểm thời gian ngẫu nhiên để tối ưu hóa gradient
        valid_start_indices = list(range(0, T_trace - K))
        sample_indices = np.random.choice(
            valid_start_indices, 
            size=min(sample_steps, len(valid_start_indices)), 
            replace=False
        )
        
        accum_loss = 0.0
        accum_ce = 0.0
        accum_v_mse = 0.0
        accum_r_mse = 0.0
        
        for t_start in sample_indices:
            optimizer.zero_grad()
            L_theta = 0.0
            
            # Vòng lặp K bước mở rộng (Công thức 14)
            for k in range(K):
                idx = t_start + k
                o_k = trace['obs'][idx].to(device)
                hat_r_k = torch.tensor(trace['rewards'][idx], dtype=torch.float32).to(device)
                hat_v_k = trace['returns'][idx].to(device)
                
                # Forward các mạng
                s_k = h_net(o_k)
                a_k, v_k = f_net(s_k)
                
                with torch.no_grad():
                    hat_a_k, _, _ = mcts.search(s_k.detach())
                    
                pred_r_k, _ = g_net(s_k, hat_a_k)
                
                # Hàm mất mát kết hợp
                step_loss, l_a, l_v, l_r = compute_joint_loss(
                    a_k, hat_a_k, v_k, hat_v_k, pred_r_k, hat_r_k
                )
                L_theta += step_loss
                accum_ce += l_a
                accum_v_mse += l_v
                accum_r_mse += l_r
                
            L_theta.backward()
            optimizer.step()
            accum_loss += L_theta.item()
            
        elapsed = time.time() - start_time
        num_updates = len(sample_indices)
        avg_loss = accum_loss / num_updates
        
        print(f"Epoch [{epoch:03d}/{epochs:03d}] ({elapsed:.1f}s) | "
              f"Cumulative Return: {total_epoch_return:.4f} | "
              f"Loss: {avg_loss:.4f} "
              f"(CE: {accum_ce/(num_updates*K):.3f}, "
              f"V_MSE: {accum_v_mse/(num_updates*K):.4f}, "
              f"R_MSE: {accum_r_mse/(num_updates*K):.4f})")
        
        # Lưu lại checkpoint tốt nhất dựa trên tổng lợi nhuận tích lũy
        if total_epoch_return > best_return:
            best_return = total_epoch_return
            torch.save({
                'h_net': h_net.state_dict(),
                'f_net': f_net.state_dict(),
                'g_net': g_net.state_dict(),
                'epoch': epoch,
                'return': best_return
            }, save_path)
            
    print("=" * 75)
    print(f">>> Hoàn tất huấn luyện toàn diện! Mô hình tối ưu nhất đã được lưu tại: {save_path}")






## 4. CHẠY  VÒNG LẶP HUẤN LUYỆN 



In [ ]:
train_full_portfoliozero(
    env=train_env,
    h_net=h_theta,
    f_net=f_theta,
    g_net=g_theta,
    mcts=mcts_agent,
    epochs=45,             # 50 Epochs để hội tụ đầy đủ
    K=5,                   # K = 5 bước unroll
    sample_steps=64        # 64 batch updates per epoch
)

print("\n>>> Hoàn thành Bước 5: Vòng lặp huấn luyện Algorithm 1 và Hàm mất mát Joint Loss (Eq. 14) đã hoạt động chính xác!")

# Đánh giá 
## 1. HÀM TÍNH TOÁN 5 CHỈ SỐ ĐÁNH GIÁ (Công thức 15 -> 19)

BẮT ĐẦU HUẤN LUYỆN TOÀN DIỆN PORTFOLIOZERO R1 (FULL IN-SAMPLE PERIOD)
- Tổng số ngày giao dịch (T): 726 ngày
- Số Epochs: 45 | Unroll K: 5 | MCTS Sims: 30
- Tối ưu hóa: Adam (lr=0.0001, weight_decay=0.0001)


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

def evaluate_portfolio_metrics(portfolio_values, daily_returns, annual_days=242):
    """
    Tính toán đúng 5 chỉ số tài chính theo Mục 4.1.4:
    LAPV, ARR, SR, MDD, CR.
    """
    p_arr = np.array(portfolio_values)
    p0 = p_arr[0]
    pT = p_arr[-1]
    T_steps = len(daily_returns)
    
    # 1. Logarithmic Accumulation of Portfolio Value (LAPV - Eq. 15)
    lapv = float(np.log(max(pT / p0, 1e-8)))
    
    # 2. Annualized Rate of Return (ARR - Eq. 16)
    arr = float((pT / p0) ** (annual_days / T_steps) - 1.0)
    
    # 3. Sharpe Ratio (SR - Eq. 17)
    r_arr = np.array(daily_returns)
    std_r = np.std(r_arr)
    sr = float((np.mean(r_arr) / (std_r + 1e-8)) * np.sqrt(annual_days))
    
    # 4. Maximum Drawdown (MDD - Eq. 18)
    peak = np.maximum.accumulate(p_arr)
    drawdowns = (peak - p_arr) / peak
    mdd = float(np.max(drawdowns))
    
    # 5. Calmar Ratio (CR = ARR / MDD - Eq. 19)
    cr = float((arr / mdd) * 100) if mdd > 1e-6 else 0.0
    
    return {
        'LAPV': round(lapv, 4),
        'SR': round(sr, 4),
        'MDD': round(mdd, 4),
        'ARR(%)': round(arr * 100, 4),
        'CR(%)': round(cr, 4)
    }



In [ ]:
# =======================================================
# 2. GIAI ĐOẠN SUY LUẬN (INFERENCE): o_t -> s_t -> a_t
# =======================================================
def run_inference(env, h_net, f_net):
    """
    Chạy suy luận trực tiếp trên tập Out-of-sample (năm 2023).
    Không sử dụng MCTS để tối đa hóa tốc độ phản xạ thị trường (Mục 3.3.4).
    """
    h_net.eval()
    f_net.eval()
    
    obs_raw, _ = env.reset()
    done = False
    
    history_pv = [env.portfolio_value]
    history_r = []
    history_weights = []
    
    with torch.no_grad():
        while not done:
            # 1. Chuyển đổi qua Representation Network: o_t -> s_t
            obs_tensor = torch.tensor(obs_raw, dtype=torch.float32).to(device)
            s_t = h_net(obs_tensor)
            
            # 2. Quyết định qua Prediction Network: s_t -> a_t (Softmax)
            a_t, _ = f_net(s_t)
            action_np = a_t.cpu().numpy()
            
            # 3. Tương tác với môi trường kiểm thử
            next_obs, reward, terminated, truncated, info = env.step(action_np)
            
            history_pv.append(info['portfolio_value'])
            history_r.append(reward)
            history_weights.append(info['weights'])
            
            obs_raw = next_obs
            done = terminated or truncated
            
    return history_pv, history_r, history_weights

# =======================================================
# 3. CHẠY BASELINE: CHIẾN LƯỢC EQUAL WEIGHT
# =======================================================
def run_equal_weight_baseline(env):
    """Chiến lược phân bổ đều tỷ trọng (1 / num_assets) làm mốc cơ sở (Benchmark)"""
    env.reset()
    done = False
    eq_action = np.ones(env.num_assets) / env.num_assets
    
    history_pv = [env.portfolio_value]
    history_r = []
    
    while not done:
        _, reward, terminated, truncated, info = env.step(eq_action)
        history_pv.append(info['portfolio_value'])
        history_r.append(reward)
        done = terminated or truncated
        
    return history_pv, history_r

# Chạy suy luận trên môi trường Test (test_env)
print(">>> Đang thực hiện Backtest Out-of-Sample 2023...")
pz_pv, pz_r, _ = run_inference(test_env, h_theta, f_theta)
ew_pv, ew_r = run_equal_weight_baseline(test_env)

# Tính toán các chỉ số thực tế
metrics_pz = evaluate_portfolio_metrics(pz_pv, pz_r, annual_days=len(pz_r))
metrics_ew = evaluate_portfolio_metrics(ew_pv, ew_r, annual_days=len(ew_r))

# =======================================================
# 4. TỔNG HỢP KẾT QUẢ & ĐỐI CHIẾU BẢNG 4 BÀI BÁO (Trang 27)
# =======================================================
comparison_data = {
    "Chiến lược / Nguồn": [
        "PortfolioZero_R1 (Mô hình huấn luyện)",
        "Equal Weight (Mô phỏng thực tế)",
        "PortfolioZero_R1 (Bài báo - Table 4)",
        "Equal Weight (Bài báo - Table 4)",
        "DDPG_R1 (Bài báo - Table 4)"
    ],
    "LAPV": [
        metrics_pz['LAPV'], 
        metrics_ew['LAPV'], 
        0.1208, 
        0.0809, 
        0.1007
    ],
    "SR": [
        metrics_pz['SR'], 
        metrics_ew['SR'], 
        0.7172, 
        0.5731, 
        0.6524
    ],
    "MDD": [
        metrics_pz['MDD'], 
        metrics_ew['MDD'], 
        0.1922, 
        0.1930, 
        0.2111
    ],
    "ARR (%)": [
        metrics_pz['ARR(%)'], 
        metrics_ew['ARR(%)'], 
        12.8443, 
        8.4228, 
        10.5928
    ],
    "CR (%)": [
        metrics_pz['CR(%)'], 
        metrics_ew['CR(%)'], 
        66.8408, 
        43.6508, 
        50.1737
    ]
}

df_results = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("          BẢNG SO SÁNH HIỆU NĂNG TẬP BIẾN ĐỘNG (VOLATILE PORTFOLIO 2023)")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

# =======================================================
# 5. TRỰC QUAN HÓA ĐƯỜNG TÍCH LŨY TÀI SẢN (LAPV CURVE)
# =======================================================
plt.figure(figsize=(12, 5))
plt.plot(np.log(np.array(pz_pv)), label='PortfolioZero_R1 (Our Test)', color='red', linewidth=1.8)
plt.plot(np.log(np.array(ew_pv)), label='Equal Weight Baseline', color='blue', linestyle='--', linewidth=1.5)
plt.title("Đường cong tích lũy giá trị tài sản logarit (LAPV) - Out-of-Sample 2023", fontsize=13)
plt.xlabel("Ngày giao dịch (Trading Days)", fontsize=11)
plt.ylabel("Log Portfolio Value (LAPV)", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\n>>> Toàn bộ quy trình từ Bước 1 đến Bước 6 đã hoàn thành xuất sắc!")